# Notebook 2 — Region annotation with **TissueTag2**, spatial distances, cortical axes & layers

### Summer School on Spatial Transcriptomics · France 2026

**Visium HD mouse brain** (fresh-frozen, 10x Genomics).

`TissueTag` lets you annotate **tissue-level regions** on a histology image — automatically (a pixel classifier + gene-expression seeds) and/or manually (an interactive drawing tool) — and then turn those region labels into **quantitative spatial features**: distances to each region, continuous anatomical **axes**, and discrete **domains** (e.g. cortical layers).

In this notebook you will:

1. Read the Visium HD image + AnnData into a `TissueTag` object.
2. **Annotate brain regions** — gene-guided seeds → Random-Forest pixel classifier → interactive refinement.
3. Compute **distances** from every spot to each annotated region.
4. Build the **pia → white-matter cortical axis**, and split the cortex into **layers**.
5. Find **genes graded along the pia→white-matter axis**.

> **Package versions.** Install `tissue_tag` from the **`main`** branch (it has the current distance/axis API used here). We follow the *teaching flow* of the `oa_update` mouse-brain tutorial, but call the `main`-branch functions:
> `run_tissuetag_visium_distance_pipeline` / `generate_grid_from_annotation` + `calculate_distance_to_annotations` + `map_annotations_to_target`, `calculate_axis`, `bin_axis`.
>
> ```bash
> pip install "git+https://github.com/DRPTSB/TissueTag2.git@main"
> ```
>
> **Live-course note.** The interactive annotator and the Random-Forest classifier need a running Jupyter/Panel server and a few minutes of compute. Every result is check-pointed to an `.h5` annotation file, so you can **load a pre-made annotation** and go straight to the quantitative parts.


## 0 · Setup

`vis_hd_aux_func.py` (in `scripts/`, from the TissueTag2 tutorial) provides the Visium HD reader and the gene-seed / classifier helpers. Keep it next to this notebook or add `scripts/` to the path.

In [ ]:
import os, sys
sys.path.append("../scripts")   # so `import vis_hd_aux_func` works from notebooks/

import numpy as np
import pandas as pd
import scanpy as sc
import panel as pn

import bin2cell as b2c
import tissue_tag as tt
import tissue_tag.annotation
import tissue_tag.io
from tissue_tag.io import TissueTagAnnotation

import vis_hd_aux_func as aux   # tutorial helpers: read_visium_hd, gene_labels_from_adata, median_filter, sk_rf_classifier

os.environ["BOKEH_ALLOW_WS_ORIGIN"] = "*"
host = "8888"   # Panel/Bokeh port; match the port in your browser when on a remote server
pn.extension()
sc.settings.set_figure_params(dpi=90, facecolor="white")

## 1 · Data

Download the 10x **Visium HD Mouse Brain (Fresh Frozen)** dataset (see `scripts/download_data.sh`):

```bash
mkdir -p data/mouse_brain && cd data/mouse_brain
BASE=https://cf.10xgenomics.com/samples/spatial-exp/3.1.1/Visium_HD_Mouse_Brain_Fresh_Frozen
curl -O $BASE/Visium_HD_Mouse_Brain_Fresh_Frozen_binned_outputs.tar.gz
curl -O $BASE/Visium_HD_Mouse_Brain_Fresh_Frozen_spatial.tar.gz
curl -O $BASE/Visium_HD_Mouse_Brain_Fresh_Frozen_tissue_image.tif   # optional full-res image
tar -xzf Visium_HD_Mouse_Brain_Fresh_Frozen_binned_outputs.tar.gz
tar -xzf Visium_HD_Mouse_Brain_Fresh_Frozen_spatial.tar.gz
```

`TissueTag` works at the **tissue** level, so we annotate on the coarse **16 µm** bins at ~0.5 px/µm (2 µm/px). The labels can later be mapped to any resolution (16/8/2 µm or a `bin2cell` single-cell object).

In [ ]:
# ---- EDIT THESE PATHS ----
DATA_DIR = "data/mouse_brain"
LIBRARY_ID = "Visium_HD_Mouse_Brain_Fresh_Frozen"
bin_resolution = 16                          # annotate on 16um bins

spaceranger_dir_path   = DATA_DIR
spaceranger_spatial_path = f"{DATA_DIR}/spatial"
mapped_image_path      = f"{DATA_DIR}/{LIBRARY_ID}_tissue_image.tif"   # only needed for use_resolution='mapped_res'

CKPT_DIR = "data/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

### 1.1 · Read the image into a TissueTag object, and the counts into AnnData

`use_resolution="hires"` uses the Space Ranger hi-res PNG (no big image download needed). For the sharpest annotation, use `use_resolution="mapped_res"` with the original full-resolution `tissue_image.tif`.

In [ ]:
tt_obj = aux.read_visium_hd(
    spaceranger_dir_path=spaceranger_dir_path,
    spaceranger_spatial_path=spaceranger_spatial_path,
    mapped_image_path=mapped_image_path,
    use_resolution="hires",      # or "mapped_res" for full-res quality
    ppm_out=0.5,                 # 0.5 px/µm == 2 µm/px, recommended for tissue-level work
    plot=True,
    bin_resolution=f"{bin_resolution:02d}",
)

In [ ]:
# AnnData at the same binning (bin2cell's Visium HD reader)
adata = b2c.read_visium(
    path=f"{DATA_DIR}/binned_outputs/square_0{bin_resolution}um/",
    spaceranger_image_path=spaceranger_spatial_path,
)
adata.var_names_make_unique()
adata

---
# Part A · Annotate brain regions

We combine three strategies:

* **Gene-expression seeds** — mark training pixels where marker genes are high (`aux.gene_labels_from_adata`).
* **Pixel classifier** — a Random-Forest on image texture/colour features fills the whole tissue (`aux.sk_rf_classifier`).
* **Manual refinement** — draw/correct regions in the interactive annotator (`tt.annotation.annotator`).


### A.1 · Define the region palette

The **order matters**: each region maps to an integer pixel value in the label image (`unassigned`=1, `isocortex`=2, …). If you extend an existing annotation, **append** new regions at the end — don't reorder.

In [ ]:
tt_obj.annotation_map = {
    'unassigned':     'yellow',
    'isocortex':      'green',
    'hippocampus':    'darkgreen',
    'olfactory':      'orange',
    'striatum':       'red',
    'thalamus':       'blue',
    'amygdala':       'lime',
    'choroid_plexus': 'gold',
    'pia':            'deepskyblue',
    'white_matter':   'white',
    'gray_matter':    'teal',
    'dentate_gyrus':  'violet',
    'layer_1':        'tan',
}

### A.2 · Gene-expression seeds for gray vs white matter

Pick marker genes and an expression threshold (top-N spots) per region. These become training labels for the classifier.

In [ ]:
gene_markers = {
    'gray_matter':  [('Gad1', 2500), ('Gad2', 2500), ('Slc17a7', 3000)],
    'white_matter': [('Mbp', 1500), ('Gfap', 500)],
}

aux.gene_labels_from_adata(
    adata=adata,
    gene_markers=gene_markers,
    tissue_tag_annotation=tt_obj,
    diameter=bin_resolution * 4,   # painting diameter (µm)
    override_labels=True,
    normalize=True,
)

aux.median_filter(tt_obj, filter_radius=bin_resolution)   # de-noise the seed labels
_ = tissue_tag.annotation.plot_labels(tt_obj, alpha=0.5)

### A.3 · Random-Forest pixel classifier

Trains on the seeded pixels using multiscale intensity/edge/texture features and predicts a region for every pixel. **~1–10 min** depending on image size.

In [ ]:
tt_obj = aux.sk_rf_classifier(tt_obj)

### A.4 · Add more gene-guided hints for manual refinement

These extra marker-based labels are **hints** (not final) to guide manual drawing of the finer regions.

In [ ]:
gene_markers = {
    'isocortex':      [('Pak7', 500), ('Myl4', 500), ('Ttc9b', 500)],
    'amygdala':       [('Acvr2a', 300)],
    'olfactory':      [('Cdhr1', 500)],
    'striatum':       [('Adora2a', 200), ('Gprin3', 200)],
    'thalamus':       [('Plekhg1', 500)],
    'choroid_plexus': [('Tcf21', 500)],
    'hippocampus':    [('Zbtb20', 500)],
}

aux.gene_labels_from_adata(
    adata=adata,
    gene_markers=gene_markers,
    tissue_tag_annotation=tt_obj,
    diameter=bin_resolution * 2,
    normalize=False,
)
_ = tissue_tag.annotation.plot_labels(tt_obj, alpha=0.25)

### A.5 · Interactive refinement  *(live only)*

Run the cell below to open the annotator: pick a region (coloured initial on the left), draw over it, then **Update** to fill. Use **Revert** to undo. `use_datashader=True` is recommended for large images.

> This needs a live Jupyter/Panel server. In a non-interactive run, skip this cell and load the checkpoint in A.6.

In [ ]:
# LIVE ONLY — opens the interactive annotator
annotator = tissue_tag.annotation.annotator(tt_obj, use_datashader=True)
pn.io.notebook.show_server(annotator, notebook_url=f"localhost:{host}")

### A.6 · Save / load the annotation checkpoint

Save your annotation to `.h5` and reload to confirm it round-trips. **In the course, start here** by loading a pre-made annotation so everyone has identical regions for the quantitative parts.

In [ ]:
anno_dir = f"{DATA_DIR}/binned_outputs/square_0{bin_resolution}um/tt_annotations"
os.makedirs(anno_dir, exist_ok=True)
anno_path = f"{anno_dir}/annotations_v1.h5"

# 💾 Save (after you've annotated)
tt_obj.save_annotation(file_path=anno_path)
print("saved", anno_path)

In [ ]:
# ⏩ Load checkpoint — run this to load a pre-made annotation instead of annotating live
tt_obj = tt.load_annotation(file_path=anno_path)
_ = tissue_tag.annotation.plot_labels(tt_obj, alpha=0.5)

---
# Part B · From regions to spatial features

We now turn region labels into numbers. `TissueTag` lays a regular **grid** over the tissue, assigns each grid point a region, and computes the **mean distance from each grid point to every region** (fast KD-tree). We then map those distances onto the Visium spots.

The `main` branch bundles these steps into `run_tissuetag_visium_distance_pipeline`, but we run them explicitly so we also get a per-spot **region label** for masking and plotting.


In [ ]:
grid_unit_size = 15   # grid spacing in microns
nhood_size     = 10   # k nearest neighbours for the mean-distance estimate

# 1) grid over the annotation, each point gets the local region label
grid_df = tt.generate_grid_from_annotation(tt_obj, grid_unit_size=grid_unit_size)

# 2) mean distance from every grid point to each region  -> L2_dist_annotation_<region> columns
grid_df = tt.calculate_distance_to_annotations(grid_df, knn=nhood_size)

grid_df = grid_df[grid_df['annotation'] != 'unassigned']
grid_df['annotation'].value_counts()

### B.1 · Map grid → Visium spots

Nearest-neighbour transfer (within `max_distance`) of the grid's region label and all distance columns onto `adata.obs`.

In [ ]:
microns_per_pixel = adata.uns['spatial'][LIBRARY_ID]['scalefactors']['microns_per_pixel']
ppm_target = 1.0 / microns_per_pixel

vis_df = tt.map_annotations_to_target(
    df_source=grid_df,
    df_target=pd.DataFrame(adata.obsm['spatial'], index=adata.obs_names, columns=["x", "y"]),
    ppm_source=1.0,               # grid is already in microns
    ppm_target=ppm_target,
    plot=True,
    max_distance=grid_unit_size * 2,
)

vis_df = vis_df.loc[adata.obs.index]
# add annotation + distance columns (everything except the x/y coords) to adata.obs
new_cols = [c for c in vis_df.columns if c not in ('x', 'y')]
adata.obs = pd.concat([adata.obs, vis_df[new_cols]], axis=1)
adata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated()]
adata.obs.head()

In [ ]:
# region labels and the raw distance fields, in tissue space
sc.set_figure_params(figsize=[5, 5], dpi=100)
sc.pl.spatial(adata, color=['annotation'], ncols=1)

l2_cols = [c for c in adata.obs.columns if c.startswith('L2_dist')]
sc.pl.spatial(adata, color=l2_cols, cmap='jet', ncols=4)

---
# Part C · Cortical axis (pia → white matter)

An **axis** turns two distance fields into one signed, normalised coordinate. `tt.calculate_axis` uses
`axis = (d[S1] − d[S2]) / (d[S1] + d[S2])`, so with `S1 = pia`, `S2 = white_matter` the value runs
**−1 at the pia surface → +1 at the white matter** — i.e. cortical depth.

We build a couple of axes and, as in the reference tutorial, **mask** each to the region where it is meaningful.

In [ ]:
def dcol(region):
    # L2 distance column name for a region
    return f"L2_dist_annotation_{region}"

# pia -> white matter (cortical depth) and isocortex -> white matter
adata.obs = tt.calculate_axis(adata.obs, feature_columns=[dcol('pia'), dcol('white_matter')], output_column='pia_to_wm')
adata.obs = tt.calculate_axis(adata.obs, feature_columns=[dcol('isocortex'), dcol('white_matter')], output_column='iso_to_wm')

# keep pia_to_wm only inside the cortex, and drop spots far from BOTH landmarks
adata.obs.loc[~adata.obs['annotation'].isin(['isocortex', 'layer_1']), 'pia_to_wm'] = np.nan
far = (adata.obs[dcol('white_matter')] > 1000) & (adata.obs[dcol('pia')] > 1000)
adata.obs.loc[far, 'pia_to_wm'] = np.nan

# iso_to_wm only near the isocortex/white-matter interface
adata.obs.loc[~adata.obs['annotation'].isin(['isocortex', 'white_matter']), 'iso_to_wm'] = np.nan
adata.obs.loc[adata.obs[dcol('white_matter')] > 300, 'iso_to_wm'] = np.nan
adata.obs.loc[adata.obs[dcol('isocortex')] > 300, 'iso_to_wm'] = np.nan

sc.set_figure_params(figsize=[5, 5], dpi=100)
sc.pl.spatial(adata, color=['pia_to_wm', 'iso_to_wm'], cmap='gist_rainbow', ncols=2)

---
# Part D · Cortical layers from the axis

Binning the continuous `pia_to_wm` axis with `tt.bin_axis` splits the cortex into ordered **domains** — an illustrative laminar segmentation (L1 → L6). Tune the cutoffs to your tissue; here we cut the −1…+1 axis into six bins.

In [ ]:
layer_labels = ['L1', 'L2/3', 'L4', 'L5', 'L6', 'L6b']
cutoffs      = [-0.6, -0.2, 0.1, 0.4, 0.7]     # len(labels) == len(cutoffs) + 1

adata.obs = tt.bin_axis(adata.obs, axis_column='pia_to_wm',
                        bin_labels=layer_labels, cutoff_values=cutoffs)

# only keep layers where the axis is defined (cortex)
adata.obs.loc[adata.obs['pia_to_wm'].isna(), 'binned_pia_to_wm'] = 'unassigned'
adata.obs['binned_pia_to_wm'] = pd.Categorical(
    adata.obs['binned_pia_to_wm'], categories=['unassigned'] + layer_labels, ordered=True)

sc.pl.spatial(adata, color='binned_pia_to_wm', ncols=1)

---
# Part E · Genes graded along the pia → white-matter axis

Which genes change smoothly with cortical depth? We normalise the data, restrict to cortical spots with a defined axis value, and rank genes by **correlation with `pia_to_wm`**. Then we visualise the top graded genes across the cortical layers.

In [ ]:
# standard normalisation for expression analysis
adata.layers['counts'] = adata.X.copy()
sc.pp.filter_cells(adata, min_counts=100)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.var_names_make_unique()

In [ ]:
# cortical spots with a defined depth
ctx = adata[adata.obs['pia_to_wm'].notna()].copy()

# correlate every gene with the pia->wm axis
import scipy.sparse as sp
X = ctx.X.toarray() if sp.issparse(ctx.X) else np.asarray(ctx.X)
axis = ctx.obs['pia_to_wm'].values.astype(float)
axis_c = axis - axis.mean()
Xc = X - X.mean(0)
denom = (np.sqrt((Xc**2).sum(0)) * np.sqrt((axis_c**2).sum()))
denom[denom == 0] = np.nan
corr = (Xc * axis_c[:, None]).sum(0) / denom

corr_s = pd.Series(corr, index=ctx.var_names).dropna().sort_values()
top_pia = corr_s.head(10).index.tolist()   # high near pia (negative axis)
top_wm  = corr_s.tail(10).index.tolist()   # high near white matter (positive axis)
print("Superficial (pia-side):", top_pia)
print("Deep (white-matter side):", top_wm)

In [ ]:
# mean expression of the graded genes across cortical layers
genes = top_pia[::-1] + top_wm
order = [l for l in layer_labels if l in ctx.obs['binned_pia_to_wm'].cat.categories]
sc.pl.matrixplot(ctx, genes, groupby='binned_pia_to_wm',
                 categories_order=order, standard_scale='var', cmap='viridis')

In [ ]:
# a few top genes painted in tissue space
sc.set_figure_params(figsize=[4, 4], dpi=100)
sc.pl.spatial(adata, color=(top_pia[:2] + top_wm[:2]), cmap='magma', ncols=4)

---
## Wrap-up & exercises

You annotated mouse-brain regions with `TissueTag`, converted them to spatial distances, built the pia→white-matter cortical axis, segmented cortical layers, and found depth-graded genes.

**Try:**
1. Re-run with a finer `grid_unit_size` (e.g. 10) or different `nhood_size` — how stable are the layers?
2. Build a **3-point axis** with `tt.calculate_axis([...三 columns...], weights=(0.2, 0.8))` (e.g. pia → gray → white).
3. Map the region labels onto the **2 µm** or **bin2cell single-cell** object from Notebook 1 (`tt.map_annotations_to_target`) and compare cell-type composition per layer.
4. Replace the correlation scan with `sc.tl.rank_genes_groups(ctx, 'binned_pia_to_wm')` to get per-layer markers.
